# F02 검수 클라이언트 (차례·개요·필요지식·위치)

엔진은 GitHub 저장소의 `ncs-review` 패키지에 있습니다. 이 노트북은 설치하고 호출만 합니다.
위에서 아래로 실행하세요. 결과는 실행마다 `결과/<모듈코드>_xxxx/` 폴더와 ZIP 하나로 저장됩니다.


In [ ]:
REPO_URL = "https://github.com/z1-won/ncs-module-test.git"  # ncs-review 저장소 주소로 바꾸세요 (@태그 또는 @브랜치 지정 가능)
%pip install -q "git+{REPO_URL}"
import ncs_review; print("ncs-review", ncs_review.__version__)


## 입력
원고와 (선택) 개발계획서를 한 번에 선택합니다. 역할은 내용으로 판별합니다. 파일명에 모듈 코드(`LM…`)가 없으면 `MODULE_CODE`를 지정하세요.


In [ ]:
from pathlib import Path
import tempfile
from google.colab import files

MODULE_CODE = ""          # 예: "LM1203020201_24v3". 비우면 원고 파일명에서 추출
ROLES = {}                # 역할 판별이 안 될 때: {"원고.hwpx": "manuscript", "계획서.hwp": "plan"}
AREAS = ("toc", "overview", "knowledge", "position")
FRONT_SECTION, BODY_SECTION = 0, 1

root = Path(tempfile.mkdtemp(prefix="ncs_inputs_"))
uploaded = files.upload()
PATHS = []
for name, content in uploaded.items():
    path = root / Path(name).name
    path.write_bytes(content)
    PATHS.append(str(path))
ROLES = {str(root / k): v for k, v in ROLES.items()}
print(PATHS)


## 검수 실행과 결과

In [ ]:
import json
from ncs_review import ReviewConfig, run_all, save_run

run = run_all(PATHS, roles=ROLES, module_code=MODULE_CODE,
              config=ReviewConfig(front_section=FRONT_SECTION, body_section=BODY_SECTION, areas=AREAS))
saved = save_run(run, "결과")
print(json.dumps(saved.summary, ensure_ascii=False, indent=2))
print(f"지적사항 {len(run.findings)}건 · 종합 {run.verdict}")
for f in run.findings:
    print(f["item_id"], "|", f["check"], "|", f["reason"])
try:
    import pandas as pd
    from IPython.display import display
    display(pd.DataFrame([{"항목": f["item_id"], "체크": f["check"], "판정": f["verdict"], "구역": f["region_path"], "이유": f["reason"]} for f in run.findings]))
except ImportError:
    pass


In [ ]:
from google.colab import files
files.download(str(saved.zip_path))
